# Interactive Trend + Pullback Backtest

This notebook includes:
- Data loading
- Strategy logic
- Interactive UI (sliders & inputs)


In [7]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Load your data
df = pd.read_csv('eth_binance_3y.csv')
df.columns = df.columns.str.lower()
df = df.dropna().reset_index(drop=True)

In [8]:
def add_indicators(df):
    df['ema50'] = df['close'].ewm(span=50).mean()
    df['ema200'] = df['close'].ewm(span=200).mean()

    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / loss
    df['rsi'] = 100 - (100 / (1 + rs))

    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    df['atr'] = ranges.max(axis=1).rolling(14).mean()

    return df

df = add_indicators(df)

In [9]:
def generate_signals(df, long_th=3, short_th=-3):
    df['signal'] = 0

    buy = (df['close'] > df['ema200']) & (df['close'] < df['ema50']) & (df['rsi'] < 40)
    sell = (df['close'] < df['ema200']) & (df['close'] > df['ema50']) & (df['rsi'] > 60)

    df.loc[buy, 'signal'] = 1
    df.loc[sell, 'signal'] = -1

    return df

In [10]:
def backtest(df, initial_balance=1000, sl_mult=1.5, tp_mult=3.0):
    balance = initial_balance
    position = 0

    for i in range(1, len(df)):
        price = df['close'].iloc[i]
        atr = df['atr'].iloc[i]

        if position == 0:
            if df['signal'].iloc[i] == 1:
                position = 1
                entry = price
                sl = entry - sl_mult * atr
                tp = entry + tp_mult * atr

            elif df['signal'].iloc[i] == -1:
                position = -1
                entry = price
                sl = entry + sl_mult * atr
                tp = entry - tp_mult * atr

        elif position == 1:
            if df['low'].iloc[i] <= sl:
                balance *= 0.99
                position = 0
            elif df['high'].iloc[i] >= tp:
                balance *= 1.02
                position = 0

        elif position == -1:
            if df['high'].iloc[i] >= sl:
                balance *= 0.99
                position = 0
            elif df['low'].iloc[i] <= tp:
                balance *= 1.02
                position = 0

    return balance

In [ ]:
# Widgets
capital = widgets.FloatText(value=1000, description='Kapital (€):')
sl_mult = widgets.FloatSlider(value=2.5, min=0.5, max=5, step=0.1, description='SL ATR')
tp_mult = widgets.FloatSlider(value=5.0, min=1, max=10, step=0.1, description='TP ATR')
run_button = widgets.Button(description='▶ Run Backtest', button_style='success')
output = widgets.Output()

In [12]:
def run_clicked(b):
    with output:
        output.clear_output()

        df2 = generate_signals(df.copy())
        result = backtest(df2, initial_balance=capital.value,
                          sl_mult=sl_mult.value,
                          tp_mult=tp_mult.value)

        print('Final Balance:', result)

run_button.on_click(run_clicked)

display(capital, sl_mult, tp_mult, run_button, output)

FloatText(value=1000.0, description='Kapital (€):')

FloatSlider(value=2.5, description='SL ATR', max=5.0, min=0.5)

FloatSlider(value=5.0, description='TP ATR', max=10.0, min=1.0)

Button(button_style='success', description='▶ Run Backtest', style=ButtonStyle())

Output()